In [3]:
!pip install arxiv

  Using cached arxiv-2.1.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached feedparser-6.0.10-py3-none-any.whl.metadata (2.3 kB)
  Using cached sgmllib3k-1.0.0-py3-none-any.whl
Using cached arxiv-2.1.0-py3-none-any.whl (11 kB)
Using cached feedparser-6.0.10-py3-none-any.whl (81 kB)


In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\HP\AppData\Local\Temp\ipykernel_16964\2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [3]:
import sys
print(sys.executable)

c:\Users\HP\Desktop\New folder (3)\.venv\Scripts\python.exe


In [5]:
!pip install --upgrade langchain langchain-core


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [5]:
wiki.name

'wikipedia'

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

vectordb = FAISS.from_documents(documents, OllamaEmbeddings(model="nomic-embed-text"))
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022AD35DD490>, search_kwargs={})

In [9]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith_search",
                      "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

In [10]:
retriever_tool.name

'langsmith_search'

In [11]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [12]:
tools=[wiki,arxiv,retriever_tool]

In [17]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\HP\\Desktop\\New folder (3)\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000012428A8BEC0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000124285EE7A0>)]

In [13]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    num_predict=512,
    timeout=120
)

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. You have access to the following tools:

{tools}

Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question"""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools)

C:\Users\HP\AppData\Local\Temp\ipykernel_16964\1211744120.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [15]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools)

C:\Users\HP\AppData\Local\Temp\ipykernel_16964\2344901266.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [38]:
response = llm.invoke("Who is Indian Prime Minister")
print(response.content)


As of my knowledge cutoff in December 2023, the current Prime Minister of India is Narendra Modi. He has been serving as the Prime Minister since May 26, 2014, and was re-elected for a second term in 2019.

However, please note that this information may have changed since my knowledge cutoff date. For the most up-to-date information, I recommend checking with a reliable news source or official government website.


In [45]:
response = llm.invoke("Why are you not giving correct answers")
print(response.content)

I'm a large language model, my responses are generated based on patterns and associations in the data I was trained on. While I strive to provide accurate and helpful information, I'm not perfect and can make mistakes.

There are several reasons why I may not be providing correct answers:

1. **Outdated training data**: My training data may not reflect the latest developments or updates in a particular field or topic.
2. **Limited domain knowledge**: While I have been trained on a vast amount of text data, my knowledge in specific domains may be limited or outdated.
3. **Misunderstanding context**: I may misinterpret the context or nuances of a question, leading to an incorrect response.
4. **Ambiguity or unclear questions**: If a question is ambiguous or unclear, I may struggle to provide an accurate answer.
5. **Error in my understanding of language**: I'm not perfect and can make mistakes when interpreting language, such as idioms, sarcasm, or figurative language.

If you suspect th

In [47]:
response = llm.invoke("richest IPL team")
print(response.content)

The richest IPL (Indian Premier League) teams can vary depending on several factors such as their brand value, revenue, and ownership structure. However, based on the latest available data, here are some of the richest IPL teams:

1. **Mumbai Indians**: Owned by Reliance Industries, Mumbai Indians have a brand value of over ₹50 billion (approximately $670 million USD). Their revenue is estimated to be around ₹2,500 crore (approximately $330 million USD) per year.
2. **Chennai Super Kings**: Owned by the Indian cricket board (BCCI), Chennai Super Kings have a brand value of over ₹40 billion (approximately $530 million USD). Their revenue is estimated to be around ₹2,000 crore (approximately $260 million USD) per year.
3. **Kolkata Knight Riders**: Owned by the Shah Rukh Khan-led Red Chillies Entertainment, Kolkata Knight Riders have a brand value of over ₹35 billion (approximately $460 million USD). Their revenue is estimated to be around ₹1,800 crore (approximately $230 million USD) pe

In [16]:
response = llm.invoke("Kanpur kaha hai ??")
print(response.content)

Kanpur ek prasiddh shahar hai Uttar Pradesh ke Madhya Pradesh mein. Yeh Kanpur Dehat district ka mukhyalay aur sabse bada shahar hai. Kanpur ki sthapna 18vi shatabdi mein hui thi aur iska naam "Kanpura" prachin samaj "Kanpoor" se liya gaya tha, jo ki "khan-poor" ke arth mein hota hai.

Kanpur ek mahatvapoorn sthal hai Bharatiya itihas mein, khaaskar 1857 ka Bharatiya swatantrata sangram mein. Is shahar mein bahut se prasiddh shiksha sansthan, vyapari aur udyog hain.

Kanpur ki sabse prasiddh aakarshanon mein se ek "All India Institute of Medical Sciences" (AIIMS) hai, jo ki Bharat ke sabse bade aur prasiddh ayurvedic sansthanon mein se ek hai. Iske alawa, Kanpur mein bahut se anya shiksha sansthan, vyapari aur udyog hain.

Kanpur ka jeevan adbhut aur samriddh hai, jo ki iski sanskriti, itihas aur aakarshanon ke liye prasiddh hai.


In [17]:
response = llm.invoke("What is Docker Container?")
print(response.content)

A Docker container is a lightweight and standalone package of software that includes everything an application needs to run, including code, libraries, frameworks, and system resources. It's like a virtual machine (VM), but much lighter and more efficient.

Here are some key characteristics of a Docker container:

1. **Lightweight**: Containers are much smaller than VMs, as they don't require a separate operating system instance.
2. **Isolated**: Each container runs in its own isolated environment, with its own file system, network stack, and process space.
3. **Portable**: Containers are highly portable across different environments, including development, testing, staging, and production.
4. **Efficient**: Containers share the host operating system's kernel, which reduces overhead and improves performance.

A Docker container typically includes:

1. **Application code**: The application's source code, compiled binaries, or other executable files.
2. **Dependencies**: Libraries, frame

In [18]:
response = llm.invoke("give me whole code to push this whole directory to github repository")
print(response.content)

Here is a simple Python script that uses the `os` and `subprocess` modules to recursively copy all files in a specified directory to a GitHub repository.

```python
import os
import subprocess

def main():
    # Specify your GitHub repository URL
    repo_url = "https://github.com/your-username/your-repo-name.git"

    # Specify the local directory you want to push
    local_dir = "/path/to/local/directory"

    # Create a new Git repository if it doesn't exist
    subprocess.run(["git", "init"], cwd=local_dir)

    # Add all files in the local directory to the Git index
    subprocess.run(["git", "add", "."], cwd=local_dir)

    # Commit the changes with a meaningful message
    commit_message = f"Initial push from {os.getcwd()}"
    subprocess.run(["git", "commit", "-m", commit_message], cwd=local_dir)

    # Link the local repository to the GitHub repository
    subprocess.run(["git", "remote", "add", "origin", repo_url], cwd=local_dir)

    # Push the changes to the remote reposito